: 

In [4]:
((11200 * 2) / 60) / 60

6.222222222222222

In [5]:
import requests
from bs4 import BeautifulSoup
import json
import os
import time

# --- Constantes ---
# URL base para construir links completos de áudio e referências internas
BASE_URL_CAMBRIDGE = "https://dictionary.cambridge.org"
# URL base para fazer as requisições das páginas das palavras
REQUEST_BASE_URL_DICTIONARY = "https://dictionary.cambridge.org/dictionary/english/"
# Arquivo para persistência dos dados
DATA_FILE = "cambridge_dictionary_data2.json"
# Headers para a requisição HTTP (use SEU User-Agent real para maior eficácia)
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36", # Atualize se necessário
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}
# Intervalo em segundos entre as requisições
REQUEST_DELAY_SECONDS = 2

# --- Funções Auxiliares de Parsing ---
def safe_get_text(element, default=""):
    """Extrai o texto de um elemento BeautifulSoup de forma segura."""
    return element.get_text() if element else default

def safe_get_attr(element, attr, default=""):
    """Extrai um atributo de um elemento BeautifulSoup de forma segura."""
    return element.get(attr, default) if element else default

# --- Funções de Persistência de Dados ---
def load_existing_data(filepath):
    """Carrega dados de um arquivo JSON, se existir."""
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except json.JSONDecodeError:
            print(f"Aviso: Arquivo de dados '{filepath}' corrompido. Iniciando com dados vazios.")
            return {}
        except Exception as e:
            print(f"Aviso: Não foi possível ler o arquivo '{filepath}'. Erro: {e}. Iniciando com dados vazios.")
            return {}
    return {}

def save_data(data, filepath):
    """Salva os dados em um arquivo JSON."""
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
    except Exception as e:
        print(f"Erro ao salvar dados em '{filepath}'. Erro: {e}")

# --- Função de Requisição HTTP ---
def fetch_word_html(word_to_search, headers):
    """Busca o conteúdo HTML da página da palavra."""
    url = f"{REQUEST_BASE_URL_DICTIONARY}{word_to_search.lower()}"
    print(f"Tentando acessar: {url}...")
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        print(f"Status Code: {response.status_code}. Acesso bem-sucedido.")
        return response.text
    except requests.exceptions.ConnectionError as e:
        print(f"Erro de Conexão para '{word_to_search}': {e}")
    except requests.exceptions.Timeout:
        print(f"Erro de Timeout para '{word_to_search}'.")
    except requests.exceptions.HTTPError as e:
        print(f"Erro HTTP para '{word_to_search}': {e.response.status_code} - {e.response.reason}")
    except requests.exceptions.RequestException as e:
        print(f"Erro na requisição para '{word_to_search}': {e}")
    except Exception as e:
        print(f"Um erro inesperado ocorreu durante a requisição para '{word_to_search}': {e}")
    return None

# --- Função Principal de Parsing (mantida da resposta anterior) ---
def parse_cambridge_entry(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    entry_data = {
        "word": "", "part_of_speech": "", "grammar": "",
        "pronunciations": {"uk": {}, "us": {}},
        "senses": [],
        "smart_vocabulary": {"topic": {}, "related_words": []}
    }
    entry_body = soup.find('div', class_='pr entry-body__el')
    if not entry_body: return entry_data # Retorna vazio se o corpo principal não for encontrado

    pos_header = entry_body.find('div', class_='pos-header')
    if pos_header:
        headword_span = pos_header.find('span', class_='hw dhw')
        entry_data['word'] = safe_get_text(headword_span)
        pos_span = pos_header.find('span', class_='pos dpos')
        entry_data['part_of_speech'] = safe_get_text(pos_span)
        gram_span = pos_header.find('span', class_='gram dgram')
        entry_data['grammar'] = safe_get_text(gram_span)

        uk_pron_span = pos_header.find('span', class_='uk dpron-i')
        if uk_pron_span:
            uk_audio_src = safe_get_attr(uk_pron_span.find('source', type='audio/mpeg'), 'src')
            # print("uk_pron_span HTML:", uk_pron_span) # Seu print para confirmar o span pai

            # Tentativa de encontrar o span do IPA
            ipa_element = uk_pron_span.find('span', class_='ipa dipa')
            
            # Log para depuração:
            # print(f"Elemento IPA com 'ipa dipa': {ipa_element}")

            if not ipa_element:
                # Se não encontrou com 'ipa dipa', tente apenas com 'ipa'
                # Isso é mais robusto se a classe 'dipa' nem sempre estiver presente ou variar
                ipa_element = uk_pron_span.find('span', class_='ipa')
                # Log para depuração:
                # print(f"Elemento IPA apenas com 'ipa': {ipa_element}")
            
            ipa_text = safe_get_text(ipa_element)
            # Se o ipa_text ainda estiver vindo com as barras, ex: "/ˈstɔː.ri/", você pode limpá-las:
            # if ipa_text.startswith('/') and ipa_text.endswith('/'):
            #    ipa_text = ipa_text.strip('/')
            # No entanto, com o seletor correto para o span interno, isso não deve ser necessário.

            entry_data['pronunciations']['uk'] = {
                'ipa': ipa_text,
                'audio': BASE_URL_CAMBRIDGE + uk_audio_src if uk_audio_src else ""
            }

        us_pron_span = pos_header.find('span', class_='us dpron-i')
        if us_pron_span:
            us_audio_src = safe_get_attr(us_pron_span.find('source', type='audio/mpeg'), 'src')
            
            ipa_element_us = us_pron_span.find('span', class_='ipa dipa')
            if not ipa_element_us:
                ipa_element_us = us_pron_span.find('span', class_='ipa')
            
            ipa_text_us = safe_get_text(ipa_element_us)

            entry_data['pronunciations']['us'] = {
                'ipa': ipa_text_us,
                'audio': BASE_URL_CAMBRIDGE + us_audio_src if us_audio_src else ""
            }
    else: # Fallback para a palavra se o cabeçalho não for encontrado
        headword_fallback = entry_body.find('span', class_='hw dhw')
        if headword_fallback: entry_data['word'] = safe_get_text(headword_fallback)

    pos_body = entry_body.find('div', class_='pos-body')
    if pos_body:
        for def_block in pos_body.find_all('div', class_='def-block ddef_block'):
            current_sense = {}
            ddef_h = def_block.find('div', class_='ddef_h')
            if ddef_h:
                epp_xref = ddef_h.find('span', class_='epp-xref')
                current_sense['cefr_level'] = safe_get_text(epp_xref)
            current_sense['definition'] = safe_get_text(def_block.find('div', class_='def ddef_d db'))
            
            examples, more_examples, see_also_terms = [], [], []
            def_body_ddef_b = def_block.find('div', class_='def-body ddef_b')
            if def_body_ddef_b:
                for ex_div in def_body_ddef_b.find_all('div', class_='examp dexamp', recursive=False):
                    examples.append(safe_get_text(ex_div.find('span', class_='eg deg')))
                
                see_xref_div = def_body_ddef_b.find('div', class_='xref see hax dxref-w')
                if see_xref_div:
                    for item_div in see_xref_div.find_all('div', class_='item lc'):
                        term_url = safe_get_attr(item_div.find('a'), 'href')
                        see_also_terms.append({
                            "term": safe_get_text(item_div.find('span', class_='x-h dx-h')),
                            "url": BASE_URL_CAMBRIDGE + term_url if term_url and not term_url.startswith('http') else term_url
                        })
            current_sense['examples'] = [ex for ex in examples if ex] # Remove exemplos vazios
            
            daccord_more_examples = def_block.find('div', class_='daccord')
            if daccord_more_examples and safe_get_text(daccord_more_examples.find('span', class_='showmore')) == "More examples":
                for li_tag in daccord_more_examples.find_all('li', class_='eg dexamp hax'):
                    more_examples.append(safe_get_text(li_tag))
            current_sense['more_examples'] = [ex for ex in more_examples if ex] # Remove exemplos vazios
            current_sense['see_also'] = see_also_terms
            
            if current_sense.get('definition') or current_sense.get('examples'):
                entry_data['senses'].append(current_sense)

    smart_vocab_div = entry_body.find('div', class_='smartt daccord')
    if smart_vocab_div:
        topic_anchor = smart_vocab_div.find('div', class_='daccord_lt').find('a') if smart_vocab_div.find('div', class_='daccord_lt') else None
        if topic_anchor:
            entry_data['smart_vocabulary']['topic'] = {
                "name": safe_get_text(topic_anchor), "url": safe_get_attr(topic_anchor, 'href')
            }
        related_words_list = smart_vocab_div.find('ul', class_='hul-u')
        if related_words_list:
            for li_tag in related_words_list.find_all('li', class_='lc'):
                word_link_tag = li_tag.find('a')
                if word_link_tag:
                    word_text = ""
                    base_span = word_link_tag.find('span', class_='base')
                    if base_span:
                        text_parts = [s.get_text() for s in base_span.find_all(True, recursive=False) if s.get_text()]
                        word_text = ' '.join(text_parts) if text_parts else safe_get_text(base_span)
                    else:
                        results_span = word_link_tag.find('span', class_='results')
                        word_text = safe_get_text(results_span) if results_span else safe_get_text(word_link_tag)
                    
                    if word_text:
                        entry_data['smart_vocabulary']['related_words'].append({
                            "word": word_text, "url": safe_get_attr(word_link_tag, 'href')
                        })
    return entry_data

# --- Lógica Principal ---
def main():
    # Defina aqui a lista de palavras que você quer processar
    words_to_process = [
        "story", "have", "elegance", "ubiquitous", "ephemeral", "serendipity", "labyrinth",
        "eloquent", "pragmatic", "resilience", "quintessential", "synergy",
        "ambiguous", # Exemplo de palavra que pode já existir ou não
        "nonexistentwordxyz" # Exemplo de palavra que provavelmente resultará em erro de busca
    ]

    all_words_data = load_existing_data(DATA_FILE)
    words_newly_collected_count = 0
    words_skipped_count = 0
    words_failed_count = 0

    print(f"--- Iniciando Coleta de Dados do Cambridge Dictionary ---")
    print(f"Carregados {len(all_words_data)} registros do arquivo '{DATA_FILE}'.")

    for word_input in words_to_process:
        normalized_key = word_input.lower().strip()
        if not normalized_key:
            continue

        # Verifica se a palavra já foi coletada E se os dados são válidos (não apenas um placeholder de erro)
        if normalized_key in all_words_data and all_words_data[normalized_key].get("word"):
            print(f"\nDados para '{normalized_key}' já existem no arquivo. Pulando.")
            words_skipped_count += 1
            continue
        elif normalized_key in all_words_data and "error" in all_words_data[normalized_key]:
             print(f"\nPalavra '{normalized_key}' resultou em erro anteriormente. Pulando nova tentativa.")
             words_skipped_count += 1
             continue


        print(f"\nProcessando nova palavra: '{normalized_key}'")
        
        html_content = fetch_word_html(normalized_key, REQUEST_HEADERS)
        
        if html_content:
            parsed_data = parse_cambridge_entry(html_content)

            words_to_process.extend([dict_word.get("word").replace(" ", "-") for dict_word in parsed_data["smart_vocabulary"]["related_words"]])

            words_to_process = list(set(words_to_process))

            # Verifica se o parsing retornou a palavra principal, indicando sucesso parcial ou total
            if parsed_data and parsed_data.get("word"):
                all_words_data[normalized_key] = parsed_data
                print(f"Dados para '{normalized_key}' coletados e adicionados.")
                words_newly_collected_count += 1
            else:
                all_words_data[normalized_key] = {
                    "error": "Falha no parsing ou estrutura da página inesperada.",
                    "original_query": normalized_key,
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                }
                print(f"Erro no parsing para '{normalized_key}'. Estrutura da página pode ter mudado ou palavra não encontrada no formato esperado.")
                words_failed_count += 1
        else:
            # Falha ao buscar HTML, registrar erro
            all_words_data[normalized_key] = {
                "error": "Falha ao buscar o conteúdo HTML da página.",
                "original_query": normalized_key,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
            }
            print(f"Erro ao buscar HTML para '{normalized_key}'.")
            words_failed_count += 1
        
        # Salva o estado atual dos dados após cada tentativa (sucesso ou falha)
        save_data(all_words_data, DATA_FILE)
        print(f"Arquivo '{DATA_FILE}' atualizado.")

        # Pausa para ser cortês com o servidor
        print(f"Aguardando {REQUEST_DELAY_SECONDS} segundos antes das próxima {len(words_to_process)} requisições...")
        time.sleep(REQUEST_DELAY_SECONDS)

    print("\n--- Coleta Concluída ---")
    print(f"Total de palavras na lista de processamento: {len(words_to_process)}")
    print(f"Palavras novas coletadas nesta sessão: {words_newly_collected_count}")
    print(f"Palavras puladas (já existentes ou com erro anterior): {words_skipped_count}")
    print(f"Palavras com falha na coleta nesta sessão: {words_failed_count}")
    print(f"Total de registros no arquivo '{DATA_FILE}': {len(all_words_data)}")
    print(f"Dados salvos em: {os.path.abspath(DATA_FILE)}")

if __name__ == "__main__":
    main()

--- Iniciando Coleta de Dados do Cambridge Dictionary ---
Carregados 0 registros do arquivo 'cambridge_dictionary_data2.json'.

Processando nova palavra: 'story'
Tentando acessar: https://dictionary.cambridge.org/dictionary/english/story...
Status Code: 200. Acesso bem-sucedido.
Dados para 'story' coletados e adicionados.
Arquivo 'cambridge_dictionary_data2.json' atualizado.
Aguardando 2 segundos antes das próxima 34 requisições...

Processando nova palavra: 'have'
Tentando acessar: https://dictionary.cambridge.org/dictionary/english/have...
Status Code: 200. Acesso bem-sucedido.
Dados para 'have' coletados e adicionados.
Arquivo 'cambridge_dictionary_data2.json' atualizado.
Aguardando 2 segundos antes das próxima 34 requisições...

Processando nova palavra: 'elegance'
Tentando acessar: https://dictionary.cambridge.org/dictionary/english/elegance...
Status Code: 200. Acesso bem-sucedido.
Dados para 'elegance' coletados e adicionados.
Arquivo 'cambridge_dictionary_data2.json' atualizado

['be-another-story',
 'lore',
 'anti-narrative',
 'bodice-ripper',
 'kompromat',
 'cautionary-tale',
 'scenario',
 'rundown',
 'legendary',
 'semi-legendary',
 'commentary',
 'another',
 'running-commentary',
 'write-something-up',
 'shaggy-dog-story',
 'in-medias-res',
 'strand',
 'horror-story',
 'anecdote',
 'backstory']

['anecdote',
 'another',
 'anti-narrative',
 'backstory',
 'be-another-story',
 'bodice-ripper',
 'cautionary-tale',
 'commentary',
 'horror-story',
 'in-medias-res',
 'kompromat',
 'legendary',
 'lore',
 'rundown',
 'running-commentary',
 'scenario',
 'semi-legendary',
 'shaggy-dog-story',
 'strand',
 'write-something-up']

In [1]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict, Optional

class CambridgeScraper:
    """
    Um scraper BÁSICO e demonstração para o Cambridge Dictionary.
    NÃO RECOMENDADO para uso em produção devido à fragilidade e termos de uso.
    """
    BASE_URL = "https://dictionary.cambridge.org/dictionary/english/"

    def get_word_definition(self, word: str) -> Optional[Dict]:
        """
        Tenta raspar a definição de uma palavra do Cambridge Dictionary.
        
        Args:
            word (str): A palavra a ser pesquisada.
            
        Returns:
            Optional[Dict]: Um dicionário com definições e exemplos, ou None.
        """
        url = f"{self.BASE_URL}{word.lower()}"
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # As classes HTML podem mudar! Isso é apenas um exemplo.
            definitions = []
            
            # Encontre o elemento principal de definição
            entry_body = soup.find('div', class_='entry-body')
            if not entry_body:
                # Tentar outra classe ou estrutura se a primeira falhar
                entry_body = soup.find('div', class_='pr entry-body__el')


            if entry_body:
                # Iterar sobre as definições
                for def_block in entry_body.find_all('div', class_='def-block'):
                    definition_text = def_block.find('div', class_='def').text.strip()
                    examples = []
                    for ex in def_block.find_all('div', class_='examp'):
                        examples.append(ex.text.strip())
                    definitions.append({
                        "definition": definition_text,
                        "examples": examples
                    })
            
            if definitions:
                return {"word": word, "definitions": definitions}
            
            return None # Nenhuma definição encontrada
            
        except requests.exceptions.RequestException as e:
            print(f"Erro ao acessar {url}: {e}")
            return None
        except AttributeError:
            print(f"Estrutura HTML diferente do esperado para '{word}'. O scraper pode precisar ser atualizado.")
            return None
        

# --- Exemplo de Uso do Scraper ---
if __name__ == "__main__":
    print("\n--- Usando o Web Scraper (Apenas para demonstração) ---")
    scraper = CambridgeScraper()
    
    word_to_scrape = "elegance"
    scraped_data = scraper.get_word_definition(word_to_scrape)
    
    if scraped_data:
        print(f"Definição raspada para '{scraped_data['word']}':")
        for def_entry in scraped_data['definitions']:
            print(f"  - {def_entry['definition']}")
            for example in def_entry['examples']:
                print(f"    Exemplo: {example}")
    else:
        print(f"Não foi possível raspar a definição para '{word_to_scrape}'.")


--- Usando o Web Scraper (Apenas para demonstração) ---
Erro ao acessar https://dictionary.cambridge.org/dictionary/english/elegance: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Não foi possível raspar a definição para 'elegance'.


In [ ]:
import requests
from bs4 import BeautifulSoup
import json # Para imprimir o dicionário de forma legível

# URL base para construir links completos de áudio e referências
BASE_URL_CAMBRIDGE = "https://dictionary.cambridge.org"

def safe_get_text(element, default=""):
    """Extrai o texto de um elemento BeautifulSoup de forma segura."""
    return element.get_text() if element else default

def safe_get_attr(element, attr, default=""):
    """Extrai um atributo de um elemento BeautifulSoup de forma segura."""
    return element.get(attr, default) if element else default

def parse_cambridge_entry(html_content):
    """
    Analisa o conteúdo HTML de uma entrada do Cambridge Dictionary e extrai os dados.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    entry_data = {
        "word": "",
        "part_of_speech": "",
        "grammar": "",
        "pronunciations": {"uk": {}, "us": {}},
        "senses": [],
        "smart_vocabulary": {"topic": {}, "related_words": []}
    }

    # Encontra o bloco principal da entrada da palavra
    # A classe 'pr entry-body__el' é a que você identificou no seu HTML
    entry_body = soup.find('div', class_='pr entry-body__el')
    if not entry_body:
        print("Erro: Bloco principal da entrada ('pr entry-body__el') não encontrado.")
        return entry_data # Retorna dados vazios ou parciais

    # --- Palavra, Classe Gramatical, Gramática ---
    pos_header = entry_body.find('div', class_='pos-header')
    if pos_header:
        headword_span = pos_header.find('span', class_='hw dhw') # Palavra principal
        entry_data['word'] = safe_get_text(headword_span)

        pos_span = pos_header.find('span', class_='pos dpos') # Classe gramatical (noun, verb, etc.)
        entry_data['part_of_speech'] = safe_get_text(pos_span)
        
        gram_span = pos_header.find('span', class_='gram dgram') # Informações gramaticais (ex: [U], [C])
        entry_data['grammar'] = safe_get_text(gram_span)

        # --- Pronúncias ---
        uk_pron_span = pos_header.find('span', class_='uk dpron-i')
        if uk_pron_span:
            uk_ipa = safe_get_text(uk_pron_span.find('span', class_='ipa dipa'))
            uk_audio_source = uk_pron_span.find('source', type='audio/mpeg')
            uk_audio_url = safe_get_attr(uk_audio_source, 'src')
            entry_data['pronunciations']['uk'] = {
                'ipa': uk_ipa,
                'audio': BASE_URL_CAMBRIDGE + uk_audio_url if uk_audio_url else ""
            }

        us_pron_span = pos_header.find('span', class_='us dpron-i')
        if us_pron_span:
            us_ipa = safe_get_text(us_pron_span.find('span', class_='ipa dipa'))
            us_audio_source = us_pron_span.find('source', type='audio/mpeg')
            us_audio_url = safe_get_attr(us_audio_source, 'src')
            entry_data['pronunciations']['us'] = {
                'ipa': us_ipa,
                'audio': BASE_URL_CAMBRIDGE + us_audio_url if us_audio_url else ""
            }
    else:
        # Tenta encontrar a palavra principal mesmo fora do pos-header (fallback)
        headword_fallback = entry_body.find('span', class_='hw dhw')
        if headword_fallback:
             entry_data['word'] = safe_get_text(headword_fallback)


    # --- Sentidos (definições, exemplos, etc.) ---
    pos_body = entry_body.find('div', class_='pos-body')
    if pos_body:
        # Cada 'def-block' geralmente representa um sentido ou um grupo de informações relacionadas
        for def_block in pos_body.find_all('div', class_='def-block ddef_block'):
            current_sense = {}

            # Nível CEFR (Common European Framework of Reference for Languages)
            ddef_h = def_block.find('div', class_='ddef_h')
            if ddef_h:
                def_info = ddef_h.find('span', class_='def-info')
                if def_info:
                    epp_xref = def_info.find('span', class_='epp-xref')
                    current_sense['cefr_level'] = safe_get_text(epp_xref)

            # Definição
            def_div = def_block.find('div', class_='def ddef_d db')
            current_sense['definition'] = safe_get_text(def_div)

            # Exemplos principais
            examples = []
            def_body_ddef_b = def_block.find('div', class_='def-body ddef_b')
            if def_body_ddef_b:
                for ex_div in def_body_ddef_b.find_all('div', class_='examp dexamp', recursive=False):
                    example_text = safe_get_text(ex_div.find('span', class_='eg deg'))
                    if example_text:
                        examples.append(example_text)
            current_sense['examples'] = examples
            
            # Mais exemplos (geralmente dentro de um acordeão)
            more_examples = []
            # O daccord pode ser filho direto do def_block ou dentro do sense_body
            daccord_more_examples = def_block.find('div', class_='daccord')
            if daccord_more_examples:
                 # Verifica se é o acordeão de "More examples"
                header_span = daccord_more_examples.find('span', class_='showmore')
                if header_span and header_span.get_text(strip=True) == "More examples":
                    for li_tag in daccord_more_examples.find_all('li', class_='eg dexamp hax'):
                        more_examples.append(safe_get_text(li_tag))
            current_sense['more_examples'] = more_examples

            # "See" (termos relacionados)
            see_also_terms = []
            if def_body_ddef_b:
                see_xref_div = def_body_ddef_b.find('div', class_='xref see hax dxref-w')
                if see_xref_div:
                    for item_div in see_xref_div.find_all('div', class_='item lc'):
                        term_span = item_div.find('span', class_='x-h dx-h')
                        term_text = safe_get_text(term_span)
                        term_link_tag = item_div.find('a')
                        term_url = safe_get_attr(term_link_tag, 'href')
                        if term_text:
                            see_also_terms.append({
                                "term": term_text,
                                "url": BASE_URL_CAMBRIDGE + term_url if term_url and not term_url.startswith('http') else term_url
                            })
            current_sense['see_also'] = see_also_terms
            
            if current_sense.get('definition') or current_sense.get('examples'): # Adiciona apenas se houver dados úteis
                entry_data['senses'].append(current_sense)

    # --- SMART Vocabulary ---
    # Localiza a seção SMART Vocabulary (pode estar em diferentes posições dependendo da página)
    smart_vocab_div = entry_body.find('div', class_='smartt daccord')
    if smart_vocab_div:
        topic_link_tag = smart_vocab_div.find('div', class_='daccord_lt')
        if topic_link_tag:
            topic_anchor = topic_link_tag.find('a')
            entry_data['smart_vocabulary']['topic'] = {
                "name": safe_get_text(topic_anchor),
                "url": safe_get_attr(topic_anchor, 'href') # URLs aqui geralmente são absolutos
            }
            
        related_words_list = smart_vocab_div.find('ul', class_='hul-u')
        if related_words_list:
            for li_tag in related_words_list.find_all('li', class_='lc'):
                word_link_tag = li_tag.find('a')
                if word_link_tag:
                    word_text = ""
                    # Tenta extrair o texto de forma mais limpa, lidando com estruturas internas
                    base_span = word_link_tag.find('span', class_='base')
                    if base_span:
                        text_parts = [s.get_text(strip=True) for s in base_span.find_all(True, recursive=False) if s.get_text(strip=True)]
                        word_text = ' '.join(text_parts)
                        if not word_text: # Fallback se a estrutura interna for diferente
                            word_text = safe_get_text(base_span)
                    else: # Fallback para o texto completo do link se 'span.base' não existir
                        results_span = word_link_tag.find('span', class_='results')
                        if results_span:
                            word_text = safe_get_text(results_span)
                        else:
                            word_text = safe_get_text(word_link_tag) # Último fallback

                    word_url = safe_get_attr(word_link_tag, 'href')
                    if word_text:
                        entry_data['smart_vocabulary']['related_words'].append({
                            "word": word_text,
                            "url": word_url # URLs aqui geralmente são absolutos
                        })
                        
    return entry_data

# --- Seu código para fazer a requisição HTTP ---
REQUEST_BASE_URL = "https://dictionary.cambridge.org/dictionary/english/"
word_to_search = "elegance"
url_to_fetch = f"{REQUEST_BASE_URL}{word_to_search.lower()}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

html_page_content = ""

try:
    print(f"Tentando acessar: {url_to_fetch} com User-Agent customizado...")
    # response = requests.get(url_to_fetch, headers=headers, timeout=15) # Timeout aumentado
    # response.raise_for_status() 

    print(f"Status Code: {response.status_code}")
    # print(f"Tamanho da Resposta (chars): {len(response.text)}")
    # print(response.text[:2000]) # Imprime uma parte maior para depuração inicial
    print("Acesso bem-sucedido! O conteúdo da página foi recebido.")
    html_page_content = response.text

except requests.exceptions.ConnectionError as e:
    print(f"Erro de Conexão: O servidor pode ter fechado a conexão ou recusado. {e}")
    print("Isso pode indicar que o servidor está bloqueando o request, mesmo com User-Agent.")
except requests.exceptions.Timeout:
    print("Erro de Timeout: A requisição demorou muito para responder.")
except requests.exceptions.HTTPError as e:
    print(f"Erro HTTP: {e.response.status_code} - {e.response.reason}")
    print("O conteúdo da página pode não ser o esperado ou pode ser uma página de erro.")
    html_page_content = e.response.text # Algumas vezes, a página de erro ainda contém HTML útil para análise do bloqueio
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro ao fazer a requisição: {e}")
    print("Verifique se o URL está correto e se o site está online.")
except Exception as e:
    print(f"Um erro inesperado ocorreu durante a requisição: {e}")


# --- Processamento do HTML ---
if html_page_content:
    print("\nIniciando o parsing do HTML...")
    extracted_data = parse_cambridge_entry(html_page_content)
    
    print("\n--- Dados Extraídos ---")
    # Usando json.dumps para uma saída formatada e legível do dicionário
    print(json.dumps(extracted_data, indent=4, ensure_ascii=False))
else:
    print("\nNenhum conteúdo HTML para analisar devido a erro na requisição.")
    print("Se desejar testar o parser com o HTML fornecido diretamente, descomente o bloco abaixo.")

    # --- Bloco para testar o parser com o HTML fornecido diretamente ---
    # print("\n--- Testando o parser com o HTML fornecido ---")
    # provided_html_snippet = """
    # <div class="pr entry-body__el"> ... (COLE SEU SNIPPET HTML AQUI) ... </div>
    # """ # Substitua pelo seu snippet completo
    # # Verifique se o snippet está completo e correto.
    # # Se o snippet for muito grande, pode ser melhor lê-lo de um arquivo.
    # if "COLE SEU SNIPPET HTML AQUI" in provided_html_snippet:
    #     print("Por favor, substitua o placeholder com seu snippet HTML para testar o parser.")
    # else:
    #     extracted_data_from_snippet = parse_cambridge_entry(provided_html_snippet)
    #     print("\n--- Dados Extraídos do Snippet ---")
    #     print(json.dumps(extracted_data_from_snippet, indent=4, ensure_ascii=False))

Tentando acessar: https://dictionary.cambridge.org/dictionary/english/elegance com User-Agent customizado...
Status Code: 200
Acesso bem-sucedido! O conteúdo da página foi recebido.

Iniciando o parsing do HTML...
def_div:  <div class="def ddef_d db">the <a class="query" href="https://dictionary.cambridge.org/dictionary/english/quality" rel="" title="quality">quality</a> of being <a class="query" href="https://dictionary.cambridge.org/dictionary/english/grace" rel="" title="graceful">graceful</a> and <a class="query" href="https://dictionary.cambridge.org/dictionary/english/attractive" rel="" title="attractive">attractive</a> in <a class="query" href="https://dictionary.cambridge.org/dictionary/english/appearance" rel="" title="appearance">appearance</a> or <a class="query" href="https://dictionary.cambridge.org/dictionary/english/behaviour" rel="" title="behaviour">behaviour</a>: </div>

--- Dados Extraídos ---
{
    "word": "elegance",
    "part_of_speech": "noun",
    "grammar": "[ 

In [19]:
import pandas as pd

pd.DataFrame([extracted_data])

,word,part_of_speech,grammar,pronunciations,senses,smart_vocabulary
0,elegance,noun,[ U ],"{'uk': {'ipa': '', 'audio': 'https://dictionar...","[{'cefr_level': 'C1', 'definition': 'the quali...",{'topic': {'name': 'Beauty and attractiveness'...


In [13]:
from bs4 import BeautifulSoup
from typing import List, Dict, Optional

class CambridgeDefinitionExtractor:
    """
    Uma classe elegante para extrair informações abrangentes de uma entrada do Cambridge Dictionary.
    Foca na robustez e extração granular de dados.
    """
    
    def __init__(self, html_content: str):
        """
        Inicializa o extrator com o conteúdo HTML da página.
        
        Args:
            html_content (str): O HTML completo da página de definição da palavra.
        """
        self.soup = BeautifulSoup(html_content, 'html.parser')

    def extract_all_data(self) -> Dict:
        """
        Extrai todos os dados relevantes de uma entrada do Cambridge Dictionary.
        
        Retorna:
            Dict: Um dicionário contendo todas as informações extraídas,
                  incluindo seções de significados, pronúncias, etc.
        """
        data = {
            "word_title": self._extract_word_title(),
            "pronunciations": self._extract_pronunciations(),
            "meanings": self._extract_meanings(),
            "phrasal_verbs_and_idioms": self._extract_phrasal_verbs_and_idioms(),
            "related_expressions": self._extract_related_expressions()
        }
        return data

    def _extract_word_title(self) -> Optional[str]:
        """Extrai o título principal da palavra."""
        # Geralmente a palavra principal aparece em um h2 ou span com uma classe específica
        title_element = self.soup.find('h2', class_='clm-font-5-responsive') or \
                        self.soup.find('div', class_='hw pos-header') # Tenta classes comuns
        if title_element:
            # Pega o texto antes de qualquer span de pronúncia ou categoria
            return title_element.find('span', class_='headword').text.strip() if title_element.find('span', class_='headword') else title_element.text.strip()
        return None

    def _extract_pronunciations(self) -> List[Dict]:
        """Extrai as informações de pronúncia (áudio e texto fonético)."""
        pronunciations = []
        # Procurar por elementos que contenham informações de pronúncia
        # As classes comuns são 'pron', 'us', 'uk', 'ipa' e áudios em 'audio_play_button'
        
        # Pode haver várias seções de pronúncia (US, UK, etc.)
        for pron_block in self.soup.find_all('span', class_=['pron-info', 'di-info', 'dpron-i']):
            pron_text_elem = pron_block.find('span', class_=['ipa', 'dipa'])
            pron_text = pron_text_elem.text.strip() if pron_text_elem else None
            
            audio_elem = pron_block.find('source', type='audio/mpeg')
            audio_url = audio_elem['src'] if audio_elem else None

            region_elem = pron_block.find('span', class_=['region', 'dregion'])
            region = region_elem.text.strip() if region_elem else None # UK, US

            if pron_text or audio_url:
                pronunciations.append({
                    "region": region,
                    "ipa": pron_text,
                    "audio_url": audio_url
                })
        return pronunciations

    def _extract_meanings(self) -> List[Dict]:
        """
        Extrai as diferentes seções de significado (blocos de definição).
        Cada significado pode ter categoria gramatical, definições, exemplos.
        """
        meanings = []
        # O corpo principal das definições é geralmente encapsulado em 'entry-body' ou 'pr entry-body__el'
        main_entry_body = self.soup.find('div', class_='entry-body')
        if not main_entry_body:
            main_entry_body = self.soup.find('div', class_='pr entry-body__el')
        
        if not main_entry_body:
            return meanings

        # Iterar sobre as seções de definição, que podem ser agrupadas por categoria (e.g., 'noun', 'verb')
        # As classes como 'pos-header' ou 'sense-block' são bons indicadores
        for pos_block in main_entry_body.find_all('div', class_='pos-body'): # Principal bloco por parte da fala
            pos_tag_elem = pos_block.find('span', class_=['posgram', 'pos'])
            pos_tag = pos_tag_elem.text.strip() if pos_tag_elem else "N/A" # Ex: noun, verb

            for sense_block in pos_block.find_all('div', class_='sense-block'):
                # Código de referência ou número da definição
                guideword_elem = sense_block.find('span', class_=['guideword', 'dsense_intro'])
                guideword = guideword_elem.text.strip() if guideword_elem else None

                # Extrair definições e exemplos dentro deste bloco de sentido
                definitions_list = []
                for def_block in sense_block.find_all('div', class_=['def-block', 'ddef_block']):
                    definition_text_elem = def_block.find('div', class_=['def', 'ddef_d'])
                    definition_text = definition_text_elem.text.strip() if definition_text_elem else "Definição não encontrada"
                    
                    # Remover o prefixo de categoria (e.g., '[ C ]') se presente
                    definition_text = self._clean_definition_text(definition_text)

                    examples = [ex.text.strip() for ex in def_block.find_all('div', class_=['examp', 'dexample'])]
                    
                    # Sinônimos e Antônimos
                    syn_ant_block = def_block.find('div', class_='synonyms') or def_block.find('div', class_='antonyms')
                    synonyms = []
                    antonyms = []
                    if syn_ant_block:
                        for item in syn_ant_block.find_all('a', class_='syn-link'):
                            synonyms.append(item.text.strip())
                        for item in syn_ant_block.find_all('a', class_='ant-link'):
                            antonyms.append(item.text.strip())


                    definitions_list.append({
                        "text": definition_text,
                        "examples": examples,
                        "synonyms": synonyms,
                        "antonyms": antonyms
                    })
                
                if definitions_list:
                    meanings.append({
                        "part_of_speech": pos_tag,
                        "guideword": guideword,
                        "definitions": definitions_list
                    })
        return meanings
    
    def _clean_definition_text(self, text: str) -> str:
        """Remove padrões indesejados do texto da definição, como '[ C ]' ou '[ T ]'."""
        import re
        # Expressão regular para remover padrões como '[ C ]', '[ T ]', '[ S ]', '[ U ]' no início da string
        # e também os elementos de áudio play button que podem estar dentro da definição.
        cleaned_text = re.sub(r'\[\s*[CSUTILB]\s*\]', '', text).strip()
        # Remove texto de botões de áudio que porventura sejam capturados
        cleaned_text = re.sub(r'[\s\S]*?audio_play_button\s*', '', cleaned_text)
        return cleaned_text


    def _extract_phrasal_verbs_and_idioms(self) -> List[Dict]:
        """Extrai phrasal verbs e expressões idiomáticas associadas à palavra."""
        expressions = []
        # As expressões podem estar em seções como 'idm-block' ou 'pv-block'
        for expr_block in self.soup.find_all('div', class_=['idm-block', 'pv-block']):
            phrase_elem = expr_block.find('span', class_=['phrase-title', 'di-title'])
            phrase = phrase_elem.text.strip() if phrase_elem else None
            
            if phrase:
                # Extrair definições e exemplos para a frase/idioma
                expr_definitions = []
                for def_block in expr_block.find_all('div', class_=['def-block', 'ddef_block']):
                    definition_text_elem = def_block.find('div', class_=['def', 'ddef_d'])
                    definition_text = definition_text_elem.text.strip() if definition_text_elem else "Definição não encontrada"
                    definition_text = self._clean_definition_text(definition_text)
                    examples = [ex.text.strip() for ex in def_block.find_all('div', class_=['examp', 'dexample'])]
                    
                    expr_definitions.append({
                        "text": definition_text,
                        "examples": examples
                    })
                
                if expr_definitions:
                    expressions.append({
                        "phrase": phrase,
                        "definitions": expr_definitions
                    })
        return expressions
    
    def _extract_related_expressions(self) -> List[Dict]:
        """Extrai expressões e palavras relacionadas (ex: "Related words" ou "More examples")."""
        related_expressions = []
        # Elementos comuns como 'related-words' ou 'more-examples'
        for related_block in self.soup.find_all('div', class_=['related-words', 'other-related-items']):
            title_elem = related_block.find('h3', class_='cdo-section-title') # Ou outra tag de título
            title = title_elem.text.strip() if title_elem else "Expressões Relacionadas"
            
            # Pegar links ou textos de expressões
            items = [li.text.strip() for li in related_block.find_all('li', class_='cdo-usage-item')]
            
            if items:
                related_expressions.append({
                    "section_title": title,
                    "items": items
                })
        return related_expressions


# --- Exemplo de Uso (Assumindo que você já tem o HTML) ---
if __name__ == "__main__":
    # Importante: O 'html_content' deve vir de uma requisição bem-sucedida,
    # seja com headers (User-Agent) ou Selenium, como discutimos.
    # Exemplo: html_content = response.text
    
    # Para demonstração, estou usando um placeholder.
    # Na prática, você faria a requisição HTTP aqui.
    print("Por favor, certifique-se de que o 'html_content' é o HTML completo da página.")
    print("Isso geralmente é obtido via 'requests.get(url, headers=...).text' ou 'driver.page_source' do Selenium.")
    
    # --- SIMULAÇÃO DE HTML (SUBSTITUA PELA SUA REQUISIÇÃO REAL) ---
    # Aqui, para fins de teste, você pode colar um HTML capturado manualmente do navegador
    # ou usar um HTML real de um teste anterior com sucesso.
    # Por exemplo, se você salvou o response.text em um arquivo:
    # with open("cambridge_elegance.html", "r", encoding="utf-8") as f:
    #     html_content_for_test = f.read()
    # OU
    # Faça a requisição aqui:
    import requests
    # Use SEU User-Agent aqui!
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    word_to_process = "elegance" # Ou "vision", "machine learning", etc.
    target_url = f"https://dictionary.cambridge.org/dictionary/english/{word_to_process}"

    html_content_for_test = ""
    try:
        response_test = requests.get(target_url, headers=headers, timeout=10)
        response_test.raise_for_status()
        html_content_for_test = response_test.text
        print(f"\nConteúdo HTML obtido com sucesso para '{word_to_process}'.")
    except requests.exceptions.RequestException as e:
        print(f"\nErro ao obter HTML para '{word_to_process}'. Por favor, verifique a requisição ou use Selenium: {e}")
        exit() # Sai se não conseguir o HTML
    
    # --- FIM DA SIMULAÇÃO ---

    extractor = CambridgeDefinitionExtractor(html_content_for_test)
    all_extracted_data = extractor.extract_all_data()

    import json
    print("\n--- Dados Extraídos ---")
    print(json.dumps(all_extracted_data, indent=2, ensure_ascii=False))

    # Exemplo de como acessar alguns dados
    print(f"\nPalavra Principal: {all_extracted_data.get('word_title')}")
    print("\nPronúncias:")
    for pron in all_extracted_data.get('pronunciations', []):
        print(f"  - Região: {pron.get('region')}, IPA: {pron.get('ipa')}, Áudio: {pron.get('audio_url')}")

    print("\nSignificados:")
    for meaning_block in all_extracted_data.get('meanings', []):
        print(f"  Parte da Fala: {meaning_block.get('part_of_speech')}")
        if meaning_block.get('guideword'):
            print(f"    Guia de Sentido: {meaning_block.get('guideword')}")
        for definition in meaning_block.get('definitions', []):
            print(f"    - Definição: {definition.get('text')}")
            if definition.get('synonyms'):
                print(f"      Sinônimos: {', '.join(definition.get('synonyms'))}")
            if definition.get('antonyms'):
                print(f"      Antônimos: {', '.join(definition.get('antonyms'))}")
            for example in definition.get('examples', []):
                print(f"      Exemplo: \"{example}\"")

    print("\nPhrasal Verbs e Expressões Idiomáticas:")
    for expr_block in all_extracted_data.get('phrasal_verbs_and_idioms', []):
        print(f"  - Frase: {expr_block.get('phrase')}")
        for definition in expr_block.get('definitions', []):
            print(f"    Definição: {definition.get('text')}")
            for example in definition.get('examples', []):
                print(f"      Exemplo: \"{example}\"")

    print("\nExpressões Relacionadas:")
    for related_block in all_extracted_data.get('related_expressions', []):
        print(f"  Seção: {related_block.get('section_title')}")
        for item in related_block.get('items', []):
            print(f"    - {item}")

Por favor, certifique-se de que o 'html_content' é o HTML completo da página.
Isso geralmente é obtido via 'requests.get(url, headers=...).text' ou 'driver.page_source' do Selenium.

Conteúdo HTML obtido com sucesso para 'elegance'.

--- Dados Extraídos ---
{
  "word_title": null,
  "pronunciations": [
    {
      "region": "uk",
      "ipa": "ˈel.ə.ɡəns",
      "audio_url": "/media/english/uk_pron/u/uke/ukele/ukelect007.mp3"
    },
    {
      "region": "us",
      "ipa": "ˈel.ə.ɡəns",
      "audio_url": "/media/english/us_pron/e/ele/elega/elegance.mp3"
    }
  ],
  "meanings": [],
  "phrasal_verbs_and_idioms": [],
  "related_expressions": []
}

Palavra Principal: None

Pronúncias:
  - Região: uk, IPA: ˈel.ə.ɡəns, Áudio: /media/english/uk_pron/u/uke/ukele/ukelect007.mp3
  - Região: us, IPA: ˈel.ə.ɡəns, Áudio: /media/english/us_pron/e/ele/elega/elegance.mp3

Significados:

Phrasal Verbs e Expressões Idiomáticas:

Expressões Relacionadas:


In [12]:
{"word": word, "definitions": definitions}

{'word': 'elegance',
 'definitions': [{'definition': 'the quality of being graceful and attractive in appearance or behaviour:',
   'examples': ['It was her natural elegance that struck me.',
    'the elegance of her clothes']}]}